# Telomerality Analysis

In [ ]:
library(dplyr)
library(stringr)
library(readr)

fasta_path <- "results/minimap2-all-vs-all/input.filtered.fa"
stopifnot(file.exists(fasta_path))

# Orientation-specific telomeric motifs (5'->3' on + strand)
left_motif  <- "CCCTAAA"  # expected at scaffold start
right_motif <- "TTTAGGG"  # expected at scaffold end
k <- nchar(left_motif)

# Detection parameters
max_mismatch <- 1L
seed_max_bp <- 2000L
max_gap_bp <- 60L
terminal_window_bp <- 1000000L

# Parallel settings (set to 1L for serial mode)
n_cores_requested <- 12L

out_dir <- "results/telomerality"
dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

read_fasta_simple <- function(path) {
  x <- readLines(path, warn = FALSE)
  hdr_idx <- which(startsWith(x, ">"))
  if (length(hdr_idx) == 0L) stop("No FASTA headers found in: ", path)

  get_seq <- function(i) {
    start <- hdr_idx[i] + 1L
    end <- if (i < length(hdr_idx)) hdr_idx[i + 1L] - 1L else length(x)
    if (start > end) return("")
    paste0(x[start:end], collapse = "")
  }

  headers <- sub("^>", "", x[hdr_idx])
  ids <- sub("[[:space:]].*$", "", headers)
  seqs <- vapply(seq_along(hdr_idx), get_seq, character(1))

  tibble(scaffold = ids, sequence = toupper(seqs))
}

hamming_dist <- function(a, b) {
  aa <- strsplit(a, "", fixed = TRUE)[[1]]
  bb <- strsplit(b, "", fixed = TRUE)[[1]]
  sum(aa != bb)
}

approx_motif_starts <- function(seq_chr, motif, max_mismatch = 1L) {
  n <- nchar(seq_chr)
  k <- nchar(motif)
  if (n < k) return(integer(0))

  starts <- seq_len(n - k + 1L)
  starts[vapply(starts, function(i) {
    kmer <- substr(seq_chr, i, i + k - 1L)
    if (grepl("[^ACGT]", kmer)) return(FALSE)
    hamming_dist(kmer, motif) <= max_mismatch
  }, logical(1))]
}

left_terminal_extent <- function(hit_starts, n, k, seed_max_bp = 2000L, max_gap_bp = 60L) {
  if (length(hit_starts) == 0L) return(0L)
  hit_starts <- sort(unique(hit_starts))
  if (hit_starts[1] > seed_max_bp) return(0L)

  cur_right <- hit_starts[1] + k - 1L
  if (length(hit_starts) > 1L) {
    for (s in hit_starts[-1]) {
      e <- s + k - 1L
      gap <- s - cur_right - 1L
      if (gap <= max_gap_bp) cur_right <- max(cur_right, e) else break
    }
  }
  as.integer(min(cur_right, n))
}

right_terminal_extent <- function(hit_starts, n, k, seed_max_bp = 2000L, max_gap_bp = 60L) {
  if (length(hit_starts) == 0L) return(0L)
  hit_starts <- sort(unique(hit_starts))
  hit_ends <- hit_starts + k - 1L

  ord <- order(hit_ends, decreasing = TRUE)
  hs <- hit_starts[ord]
  he <- hit_ends[ord]
  if ((n - he[1] + 1L) > seed_max_bp) return(0L)

  cur_left <- hs[1]
  if (length(hs) > 1L) {
    for (i in 2:length(hs)) {
      s <- hs[i]
      e <- he[i]
      gap <- cur_left - e - 1L
      if (gap <= max_gap_bp) cur_left <- min(cur_left, s) else break
    }
  }
  as.integer(max(0L, n - cur_left + 1L))
}

get_internal_motif_positions <- function(seq_chr, left_bp, right_bp) {
  n <- nchar(seq_chr)
  interior_start <- left_bp + 1L
  interior_end <- n - right_bp
  if (interior_end < interior_start) return(list(ccctaaa = integer(0), tttaggg = integer(0)))

  interior <- substr(seq_chr, interior_start, interior_end)
  
  # Find CCCTAAA positions
  m_ccctaaa <- gregexpr("CCCTAAA", interior, perl = TRUE)[[1]]
  ccctaaa_pos <- if (length(m_ccctaaa) == 1L && m_ccctaaa[1] == -1L) {
    integer(0)
  } else {
    as.integer(m_ccctaaa + interior_start - 1L)  # convert back to full scaffold coords
  }
  
  # Find TTTAGGG positions
  m_tttaggg <- gregexpr("TTTAGGG", interior, perl = TRUE)[[1]]
  tttaggg_pos <- if (length(m_tttaggg) == 1L && m_tttaggg[1] == -1L) {
    integer(0)
  } else {
    as.integer(m_tttaggg + interior_start - 1L)  # convert back to full scaffold coords
  }
  
  list(ccctaaa = ccctaaa_pos, tttaggg = tttaggg_pos)
}

analyze_one_scaffold <- function(scaffold_name, seq_chr, len_bp) {
  win_bp <- as.integer(min(terminal_window_bp, len_bp))

  left_seq <- substr(seq_chr, 1L, win_bp)
  left_hits <- approx_motif_starts(left_seq, motif = left_motif, max_mismatch = max_mismatch)
  left_bp <- left_terminal_extent(left_hits, n = win_bp, k = k, seed_max_bp = seed_max_bp, max_gap_bp = max_gap_bp)

  right_start <- as.integer(max(1L, len_bp - win_bp + 1L))
  right_seq <- substr(seq_chr, right_start, len_bp)
  right_hits <- approx_motif_starts(right_seq, motif = right_motif, max_mismatch = max_mismatch)
  right_bp <- right_terminal_extent(right_hits, n = win_bp, k = k, seed_max_bp = seed_max_bp, max_gap_bp = max_gap_bp)

  motif_pos <- get_internal_motif_positions(seq_chr, left_bp = left_bp, right_bp = right_bp)
  n_ccctaaa <- length(motif_pos$ccctaaa)
  n_tttaggg <- length(motif_pos$tttaggg)
  total_internal <- n_ccctaaa + n_tttaggg

  tibble(
    scaffold = scaffold_name,
    length_bp = as.integer(len_bp),
    left_telomere_bp = as.integer(left_bp),
    right_telomere_bp = as.integer(right_bp),
    left_pct = 100 * left_bp / len_bp,
    right_pct = 100 * right_bp / len_bp,
    left_orientation_ok = left_bp > 0L,
    right_orientation_ok = right_bp > 0L,
    left_terminal_ok = left_bp > 0L,
    right_terminal_ok = right_bp > 0L,
    ccctaaa_internal_hits = as.integer(n_ccctaaa),
    tttaggg_internal_hits = as.integer(n_tttaggg),
    internal_exact_telomeric_hits = as.integer(total_internal),
    telomeric_status = dplyr::case_when(
      left_bp > 0 & right_bp > 0 ~ "both_ends",
      left_bp > 0 & right_bp == 0 ~ "left_only",
      left_bp == 0 & right_bp > 0 ~ "right_only",
      TRUE ~ "none_detected"
    )
  )
}

scaf <- read_fasta_simple(fasta_path) %>%
  mutate(
    sequence = str_replace_all(sequence, "[^ACGT]", "N"),
    length_bp = nchar(sequence)
  )

n_scaf <- nrow(scaf)
avail_cores <- parallel::detectCores(logical = FALSE)
if (is.na(avail_cores) || avail_cores < 1L) avail_cores <- 1L
n_cores <- max(1L, min(as.integer(n_cores_requested), n_scaf, avail_cores))

idx <- seq_len(n_scaf)

if (.Platform$OS.type == "unix" && n_cores > 1L) {
  tel_list <- parallel::mclapply(
    idx,
    function(i) analyze_one_scaffold(scaf$scaffold[i], scaf$sequence[i], scaf$length_bp[i]),
    mc.cores = n_cores,
    mc.preschedule = TRUE
  )
} else {
  tel_list <- lapply(
    idx,
    function(i) analyze_one_scaffold(scaf$scaffold[i], scaf$sequence[i], scaf$length_bp[i])
  )
}

telomere_tbl <- bind_rows(tel_list) %>%
  arrange(desc(length_bp))

print(telomere_tbl, n = nrow(telomere_tbl))
cat("Using cores:", n_cores, "(requested:", n_cores_requested, ", available:", avail_cores, ")\n")

run_stamp <- format(Sys.time(), "%Y-%m-%d_%H%M%S")
csv_path <- file.path(out_dir, paste0("telomerality_", run_stamp, ".csv"))

write.csv(telomere_tbl, csv_path, row.names = FALSE)

cat("Saved:\n", csv_path, "\n")

## Prepare for Circos

In [ ]:
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd

OUT_DIR = Path('/home/apettersson/projects/LPED-assembly-QC/results/telomerality')
OUT_DIR.mkdir(parents=True, exist_ok=True)
STRICT_LATEST = OUT_DIR / 'telomere_centromere_strict_latest.csv'

# ---- Tunable, focused confidence thresholds ----
TEL_HIGH_BP = 500
TEL_MEDIUM_BP = 100
CENT_HIGH_ENRICHMENT = 10.0
CENT_MEDIUM_ENRICHMENT = 4.0

if 'res_df' not in globals() or not isinstance(res_df, pd.DataFrame) or res_df.empty:
    if not STRICT_LATEST.exists():
        raise FileNotFoundError(f'Missing strict results file: {STRICT_LATEST}')
    base_df = pd.read_csv(STRICT_LATEST)
else:
    # Prefer strict schema if currently in kernel
    if {'left_telomere_bp', 'right_telomere_bp', 'centromere_call'}.issubset(res_df.columns):
        base_df = res_df.copy()
    elif STRICT_LATEST.exists():
        base_df = pd.read_csv(STRICT_LATEST)
    else:
        raise ValueError('No suitable strict telomere/centromere table available.')

required_cols = [
    'scaffold', 'length_bp', 'left_telomere_bp', 'right_telomere_bp',
    'telomeric_status', 'centromere_call', 'centromere_start_bp', 'centromere_end_bp',
    'centromere_top_window_hits', 'centromere_top_window_expected_hits',
]
missing = [c for c in required_cols if c not in base_df.columns]
if missing:
    raise KeyError(f'Missing required columns in strict table: {missing}')

def classify_tel_conf(bp):
    if pd.isna(bp) or float(bp) <= 0:
        return 'none'
    bp = float(bp)
    if bp >= TEL_HIGH_BP:
        return 'high'
    if bp >= TEL_MEDIUM_BP:
        return 'medium'
    return 'low'

def classify_cent_conf(call, enrichment):
    if str(call) == 'none' or pd.isna(enrichment):
        return 'none'
    if enrichment >= CENT_HIGH_ENRICHMENT:
        return 'high'
    if enrichment >= CENT_MEDIUM_ENRICHMENT:
        return 'medium'
    return 'low'

df = base_df.copy()

# Telomere coordinates (1-based closed)
df['left_tel_start_bp'] = np.where(df['left_telomere_bp'] > 0, 1, np.nan)
df['left_tel_end_bp'] = np.where(df['left_telomere_bp'] > 0, df['left_telomere_bp'], np.nan)
df['right_tel_start_bp'] = np.where(df['right_telomere_bp'] > 0, df['length_bp'] - df['right_telomere_bp'] + 1, np.nan)
df['right_tel_end_bp'] = np.where(df['right_telomere_bp'] > 0, df['length_bp'], np.nan)

# Confidence calls
df['left_tel_confidence'] = df['left_telomere_bp'].apply(classify_tel_conf)
df['right_tel_confidence'] = df['right_telomere_bp'].apply(classify_tel_conf)

exp = pd.to_numeric(df['centromere_top_window_expected_hits'], errors='coerce')
obs = pd.to_numeric(df['centromere_top_window_hits'], errors='coerce')
df['centromere_enrichment'] = np.where(exp > 0, obs / exp, np.nan)
df['centromere_confidence'] = [
    classify_cent_conf(call, enr)
    for call, enr in zip(df['centromere_call'], df['centromere_enrichment'])
]

def telomeres_high_status(lc, rc):
    if lc == 'high' and rc == 'high':
        return 'both_high'
    if lc == 'high' or rc == 'high':
        return 'one_high'
    if lc != 'none' or rc != 'none':
        return 'detected_not_high'
    return 'none_detected'

df['telomere_high_conf_status'] = [
    telomeres_high_status(lc, rc)
    for lc, rc in zip(df['left_tel_confidence'], df['right_tel_confidence'])
]
df['has_high_conf_centromere'] = df['centromere_confidence'] == 'high'

# Sort scaffold order hap1_chr1..hap2_chr6
tmp = df['scaffold'].str.extract(r'hap([12])_chr([1-6])')
df['_h'] = pd.to_numeric(tmp[0], errors='coerce').fillna(99).astype(int)
df['_c'] = pd.to_numeric(tmp[1], errors='coerce').fillna(99).astype(int)
df = df.sort_values(['_h', '_c', 'scaffold']).drop(columns=['_h', '_c']).reset_index(drop=True)

# Focused summary table (one row per chromosome)
summary_cols = [
    'scaffold', 'length_bp', 'telomeric_status', 'telomere_high_conf_status',
    'left_telomere_bp', 'left_tel_start_bp', 'left_tel_end_bp', 'left_tel_confidence',
    'right_telomere_bp', 'right_tel_start_bp', 'right_tel_end_bp', 'right_tel_confidence',
    'centromere_call', 'centromere_start_bp', 'centromere_end_bp', 'centromere_confidence',
    'centromere_enrichment',
]
focused_summary = df[summary_cols].copy()

# Long-form regions table for circos overlays
region_rows = []
for _, r in focused_summary.iterrows():
    sc = r['scaffold']
    if pd.notna(r['left_tel_start_bp']) and pd.notna(r['left_tel_end_bp']):
        region_rows.append({
            'scaffold': sc,
            'feature': 'telomere',
            'side': 'left',
            'start_bp': int(r['left_tel_start_bp']),
            'end_bp': int(r['left_tel_end_bp']),
            'length_bp': int(r['left_telomere_bp']),
            'confidence': r['left_tel_confidence'],
        })
    if pd.notna(r['right_tel_start_bp']) and pd.notna(r['right_tel_end_bp']):
        region_rows.append({
            'scaffold': sc,
            'feature': 'telomere',
            'side': 'right',
            'start_bp': int(r['right_tel_start_bp']),
            'end_bp': int(r['right_tel_end_bp']),
            'length_bp': int(r['right_telomere_bp']),
            'confidence': r['right_tel_confidence'],
        })
    if str(r['centromere_call']) != 'none' and pd.notna(r['centromere_start_bp']) and pd.notna(r['centromere_end_bp']):
        region_rows.append({
            'scaffold': sc,
            'feature': 'centromere',
            'side': 'internal',
            'start_bp': int(r['centromere_start_bp']),
            'end_bp': int(r['centromere_end_bp']),
            'length_bp': int(r['centromere_end_bp'] - r['centromere_start_bp'] + 1),
            'confidence': r['centromere_confidence'],
        })

regions_df = pd.DataFrame(region_rows, columns=['scaffold', 'feature', 'side', 'start_bp', 'end_bp', 'length_bp', 'confidence'])

# High-confidence subset (recommended first pass for circos)
regions_high_df = regions_df[regions_df['confidence'] == 'high'].copy() if not regions_df.empty else regions_df.copy()

# BED-like export (0-based start, 1-based end)
if not regions_df.empty:
    bed_df = regions_df.copy()
    bed_df['start0'] = bed_df['start_bp'] - 1
    bed_df['name'] = bed_df['feature'] + '_' + bed_df['side'] + '_' + bed_df['confidence']
    score_map = {'high': 1000, 'medium': 600, 'low': 250, 'none': 0}
    bed_df['score'] = bed_df['confidence'].map(score_map).fillna(0).astype(int)
    bed_df['strand'] = '.'
    bed_export = bed_df[['scaffold', 'start0', 'end_bp', 'name', 'score', 'strand']]
else:
    bed_export = pd.DataFrame(columns=['scaffold', 'start0', 'end_bp', 'name', 'score', 'strand'])

# Save outputs
stamp = datetime.now().strftime('%Y-%m-%d_%H%M%S')

summary_csv = OUT_DIR / f'telomere_centromere_focused_summary_{stamp}.csv'
summary_latest = OUT_DIR / 'telomere_centromere_focused_summary_latest.csv'
regions_csv = OUT_DIR / f'telomere_centromere_regions_for_circos_{stamp}.csv'
regions_latest = OUT_DIR / 'telomere_centromere_regions_for_circos_latest.csv'
high_regions_csv = OUT_DIR / f'telomere_centromere_regions_highconf_{stamp}.csv'
high_regions_latest = OUT_DIR / 'telomere_centromere_regions_highconf_latest.csv'
bed_path = OUT_DIR / f'telomere_centromere_regions_for_circos_{stamp}.bed'
bed_latest = OUT_DIR / 'telomere_centromere_regions_for_circos_latest.bed'

focused_summary.to_csv(summary_csv, index=False)
focused_summary.to_csv(summary_latest, index=False)
regions_df.to_csv(regions_csv, index=False)
regions_df.to_csv(regions_latest, index=False)
regions_high_df.to_csv(high_regions_csv, index=False)
regions_high_df.to_csv(high_regions_latest, index=False)
bed_export.to_csv(bed_path, sep='\t', index=False, header=False)
bed_export.to_csv(bed_latest, sep='\t', index=False, header=False)

print('Saved focused outputs:')
print(f'  {summary_csv}')
print(f'  {summary_latest}')
print(f'  {regions_csv}')
print(f'  {regions_latest}')
print(f'  {high_regions_csv}')
print(f'  {high_regions_latest}')
print(f'  {bed_path}')
print(f'  {bed_latest}')
print()
print('Chromosomes with both telomeres high confidence:')
print(focused_summary.loc[focused_summary['telomere_high_conf_status'] == 'both_high', ['scaffold', 'left_telomere_bp', 'right_telomere_bp']].to_string(index=False))
print()
print('Focused summary preview:')
focused_summary[['scaffold', 'telomeric_status', 'telomere_high_conf_status', 'left_telomere_bp', 'left_tel_confidence', 'right_telomere_bp', 'right_tel_confidence', 'centromere_start_bp', 'centromere_end_bp', 'centromere_confidence']]
